In [68]:
def karatsuba(x, y):
    if x < 10 or y < 10:
        return x * y

    n = max(len(str(x)), len(str(y)))
    m = n // 2

    high_x, low_x = divmod(x, 10**m)
    high_y, low_y = divmod(y, 10**m)

    z0 = karatsuba(low_x, low_y)
    z2 = karatsuba(high_x, high_y)
    z1 = karatsuba(low_x + high_x, low_y + high_y) - z0 - z2

    return z2 * (10**(2*m)) + z1 * (10**m) + z0


def karatsuba_three(a, b, c):
    return karatsuba(karatsuba(a, b), c)

In [69]:
import sympy as sy

def toom3(num1, num2):


        num1, num2 = int(num1), int(num2)
        base = 10000
        i = max(int(sy.log(num1, base)) // 3, int(sy.log(num2, base)) // 3) + 1
        B = base ** i

        d, m0 = divmod(num1, B)
        m2, m1 = divmod(d, B)

        d, n0 = divmod(num2, B)
        n2, n1 = divmod(d, B)

        po = m0 + m2
        p0 = m0
        p1 = po + m1
        p_1 = po - m1
        p_2 = (p_1 + m2) * 2 - m0
        pmax = m2

        qo = n0 + n2
        q0 = n0
        q1 = qo + n1
        q_1 = qo - n1
        q_2 = (q_1 + n2) * 2 - n0
        qmax = n2

        r0 = p0 * q0
        r1 = p1 * q1
        r_1 = p_1 * q_1
        r_2 = p_2 * q_2
        rmax = pmax * qmax

        a1 = [
            [1, 0, 0, 0, 0],
            [1, 1, 1, 1, 1],
            [1, -1, 1, -1, 1],
            [1, -2, 4, -8, 16],
            [0, 0, 0, 0, 1]
        ]

        a1 = sy.Matrix(a1)
        a1_I = a1.inv()

        a2 = sy.Matrix([r0, r1, r_1, r_2, rmax])
        ans = a1_I * a2

        return sum((ans[i, 0] * (B) ** i for i in range(len(ans))))

def toom3_three(a, b, c):
    return toom3(toom3(a, b), c)

In [70]:
import numpy as np

def fft_three(a, b, c):
    A = np.array(list(map(int, str(a)[::-1])))
    B = np.array(list(map(int, str(b)[::-1])))
    C = np.array(list(map(int, str(c)[::-1])))

    n = len(A) + len(B) + len(C)
    size = 1
    while size < n:
        size *= 2

    FA = np.fft.fft(A, size)
    FB = np.fft.fft(B, size)
    FC = np.fft.fft(C, size)

    FD = FA * FB * FC
    result = np.fft.ifft(FD).real
    result = np.round(result).astype(int)

    carry = 0
    for i in range(len(result)):
        total = result[i] + carry
        carry = total // 10
        result[i] = total % 10

    while len(result) > 1 and result[-1] == 0:
        result = result[:-1]

    return int("".join(map(str, result[::-1])))

In [71]:
import time
import random

def rand_big(n):
    return int("".join(str(random.randint(0, 9)) for _ in range(n)))

def test_once(a, b, c):
    res_k = karatsuba_three(a, b, c)
    res_t = toom3_three(a, b, c)
    res_f = fft_three(a, b, c)
    res_true = a * b * c

    return (
        res_k == res_true,
        res_t == res_true,
        res_f == res_true
    )


def benchmark():
    sizes = [50, 100, 200]
    num_tests = 100

    for s in sizes:

        total_k, total_t, total_f = 0, 0, 0
        correct_k, correct_t, correct_f = 0, 0, 0

        for _ in range(num_tests):
            a, b, c = rand_big(s), rand_big(s), rand_big(s)

            start = time.time()
            res_k = karatsuba_three(a, b, c)
            total_k += time.time() - start

            start = time.time()
            res_t = toom3_three(a, b, c)
            total_t += time.time() - start

            start = time.time()
            res_f = fft_three(a, b, c)
            total_f += time.time() - start

            true_val = a * b * c
            if res_k == true_val:
                correct_k += 1
            if res_t == true_val:
                correct_t += 1
            if res_f == true_val:
                correct_f += 1

        avg_k = total_k / num_tests
        avg_t = total_t / num_tests
        avg_f = total_f / num_tests

        print(f"Karatsuba: avg={avg_k:.6f}s, correct={correct_k}/{num_tests}")
        print(f"Toom-3   : avg={avg_t:.6f}s, correct={correct_t}/{num_tests}")
        print(f"FFT      : avg={avg_f:.6f}s, correct={correct_f}/{num_tests}")


def single_test():
    a = 12345678901234567890
    b = 98765432109876543210
    c = 11111111111111111111

    res1 = karatsuba_three(a, b, c)
    res2 = toom3_three(a, b, c)
    res3 = fft_three(a, b, c)
    res_true = a * b * c

    print("Karatsuba:", res1)
    print("Toom-3   :", res2)
    print("FFT      :", res3)

    print(res1 == res_true)
    print(res2 == res_true)
    print(res3 == res_true)


if __name__ == "__main__":
    single_test()
    benchmark()

Karatsuba: 13548070126335755024725228199972903859751392910987637385900
Toom-3   : 13548070126335755024725228199972903859751392910987637385900
FFT      : 13548070126335755024725228199972903859751392910987637385900
True
True
True
Karatsuba: avg=0.000554s, correct=100/100
Toom-3   : avg=0.005873s, correct=100/100
FFT      : avg=0.000181s, correct=100/100
Karatsuba: avg=0.001790s, correct=100/100
Toom-3   : avg=0.006152s, correct=100/100
FFT      : avg=0.000300s, correct=100/100
Karatsuba: avg=0.005380s, correct=100/100
Toom-3   : avg=0.006280s, correct=100/100
FFT      : avg=0.000452s, correct=100/100
